# Simulation of satellite-detected hailstorms in the Australian tropics

In the Australian tropics, satellites and proxies often see indications of hail or hail-prone conditions, yet the actual occurrance of hail at the surface is uncertain given sparse observations. Here I use WRF to simulate hailstorms where satellite retrievals detected the presence of hail (aloft). The simulations are run using four microphysics schemes.

In [1]:
%cd ~/git/aus_tropics_hail/

/home/561/tr2908/git/aus_tropics_hail


In [ ]:
import json
import sys

sys.path.append('../xarray_parcel/')

import glob

import cartopy.crs as ccrs
import dask
import matplotlib.pyplot as plt
import modules.nth_hail as nh
import modules.wrf_metadata as wm
import numpy as np
import pandas as pd
import xarray
from dask.distributed import Client


In [ ]:
client = Client(n_workers=24, threads_per_worker=1)
client

## Environment

In [ ]:
!module list

In [ ]:
!python --version

## MY2 modification

WRF was stopping during runs using the MY2 microphysics scheme. When I asked Jason Milbrandt about this he wrote:

`Regarding MY2, that abort message has burned me many times.  It is a forced abort designed to give that warning message if the input temperature is unreasonable.  What is likely happening is that it is in a strong updraft and the intermediate adiabatic cooling due to ascent (after the dynamics but before the microphysics) results in a temporarily very cold temperature.  Several years ago I was running a squall line case in WRF with several other MP schemes; I put print statements in those other codes and similar temperatures appeared in those schemes also.  Ultimately (by the end of the time step) the saturation adjustment and diabetic heating returns the temperature back to sane values.  In short, it probably is not a problem in MY2 (at least that’s my guess).  The fix would be either to use a smaller time step or just comment out the STOP statement in the code.`

So, I have commented out these lines in WRF's `WRF/WRF/phys/module_mp_milbrandt2mom.F` file:

```
!-----                                                                                                                                                                  
!  if ( T(i,k)<173. .or. T(i,k)>323.) then                            !** DEBUG **                                                                                      
!     print*, '** STOPPING IN MICROPHYSICS: (Part 2, end) **'         !** DEBUG **                                                                                      
!     print*, '** i,k,T [K]: ',i,k,T(i,k)                             !** DEBUG **                                                                                      
!     stop                                                            !** DEBUG **                                                                                      
!  endif                                                              !** DEBUG **                                                                                      
!=====                                                                                  
```

## Settings

In [ ]:
detecs_file = 'data/GPM_2014_2024_hail_events_P20_KIMB.nc'                # Bang & Cecil 2019 hail detections.
sims_dir = '/g/data/li18/tr2908/aus_tropics_hail/WRF_v4.6.0/simulations/' # Directory where simulations are held.
wrf_dir = '/g/data/li18/tr2908/aus_tropics_hail/WRF_v4.6.0/template/'     # Directory with compiled WRF.
namelist_dir = 'namelists/'                                               # Directory with template namelists.
prob_req = 0.5                                                            # Hail probability required to include event.
hail_threshold = 20                                                       # Threshold to count as a hail event.

## Setup

In [ ]:
plt.show()                                                       # Start the plotting engine.
plt.rcParams['font.size'] = 12                                   # Font size for plots.
plt.rcParams['axes.formatter.useoffset'] = False                 # Don't use offsets in plots.
_ = dask.config.set({'array.slicing.split_large_chunks': False}) # Allow for large dask chunks.
results = {}                                                     # Dictionary for reportable results.

## Event selection

Events to simulate are selected from data provided by Sarah Bang, with locations and times of satellite-detected hailstorms. We look for "good" detections in northern Australia with a hail probability over 50%.

In [ ]:
hail_detections = xarray.open_dataset(detecs_file).to_pandas()
hail_detections = hail_detections[hail_detections.DCflag == 1] # Select storms not flagged for snow/surface features.
hail_detections = hail_detections[hail_detections.p_hail_BC2019 > prob_req] # Select only probable detections.
#hail_detections = hail_detections[hail_detections.year < 2024] # ERA5 catalogue is only updated to Sep 2023 for now.  # noqa: PLR2004
#hail_detections = hail_detections[np.logical_not(hail_detections.year == 2023, hail_detections.month > 9)]  # noqa: PLR2004
hail_detections = hail_detections.reset_index(drop=True)
hail_detections['minute'] = np.floor(60 * (hail_detections.hour % 1))
hail_detections['hour'] = np.floor(hail_detections.hour)
print(f'The dataset contains {len(hail_detections)} hail detections.')
results['number_detections'] = len(hail_detections)

We simulate the hour before the event, the hour of the event and the hour after the event, so each event has 3 hours of simulation time. The start time is shifted back 12 hours to give a 12 hour spin-up time. Total simulation time is then 15 hours per event.

In [ ]:
hail_detections['event_time'] = (
    hail_detections.year.astype(int).astype(str).str.zfill(2)
    + '-'
    + hail_detections.month.astype(int).astype(str).str.zfill(2)
    + '-'
    + hail_detections.day.astype(int).astype(str).str.zfill(2)
    + ' '
    + hail_detections.hour.astype(int).astype(str).str.zfill(2)
    + ':'
    + hail_detections.minute.astype(int).astype(str)
)
hail_detections['event_time'] = pd.to_datetime(hail_detections.event_time, format='%Y-%m-%d %H:%M')
hail_detections['start_time'] = (hail_detections.event_time - np.timedelta64(13, 'h')).dt.floor('h')
hail_detections['end_time'] = (hail_detections.event_time + np.timedelta64(1, 'h')).dt.ceil('h')
hail_detections['start_time'] = hail_detections.start_time.dt.strftime('%Y-%m-%d_%H:%M:00')
hail_detections['end_time'] = hail_detections.end_time.dt.strftime('%Y-%m-%d_%H:%M:00')

In [ ]:
hail_detections

Here is a map showing the locations of the hail detections we study here as red dots.

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(12,8))
ax.scatter(hail_detections.longitude, hail_detections.latitude, transform=ccrs.PlateCarree(), color='red')
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, alpha=0.5)
gl.top_labels = gl.right_labels = False
ax.coastlines()
plt.show()

## Simulation computation

The following code can be run to create a directory for each simulation and set up all the namelist files to use the correct locations and times.

In [ ]:
for _, row in hail_detections.iterrows():
    nh.set_up_WRF(lat=row.latitude, lon=row.longitude, year=row.year, month=row.month, day=row.day,
                  hour=row.hour, minute=row.minute, start_time=row.start_time, end_time=row.end_time,
                  wrf_dir=wrf_dir, sims_dir=sims_dir, namelist_dir=namelist_dir)

To run WPS for each simulation:

```
cd /g/data/li18/tr2908/aus_tropics_hail/WRF_v4.6.0/simulations/
for i in lat*; do cd $i/WPS; echo `pwd`; qsub ~/git/aus_tropics_hail/scripts/run_WPS.sh; cd ../../; done
```

Once WPS has run on all, then run WRF:

```
cd /g/data/li18/tr2908/aus_tropics_hail/WRF_v4.6.0/simulations/
for i in lat*/WRF/*; do cd $i; echo `pwd`; qsub ~/git/aus_tropics_hail/scripts/run_WRF.sh; cd ../../../; done
```

To remove spin-up times:

```
for i in lat*/WRF/*; do cd $i; echo $i; ls -1 wrfout_d02* | sort | head -n -3 | xargs rm -f; cd ../../../; done
for i in lat*/WRF/*; do cd $i; echo $i; ls -1 wrfout_d03* | sort | head -n -3 | xargs rm -f; cd ../../../; done
```

It's also safe to remove all wrfout_d01 files to save space.

After WRF has finished, then run two scripts to calculate basic properties for the last three files (leaving out spin up):

```
cd /g/data/li18/tr2908/aus_tropics_hail/WRF_v4.6.0/simulations/
for i in lat*/WRF/*; do cd $i; echo `pwd`; qsub ~/git/aus_tropics_hail/scripts/process_WRF_basic.sh; cd ../../../; done
```

Run a script to calculate convective properties:

```
cd /g/data/li18/tr2908/aus_tropics_hail/WRF_v4.6.0/simulations/
for i in lat*/WRF/*; do cd $i; echo `pwd`; qsub ~/git/aus_tropics_hail/scripts/process_WRF_conv.sh; cd ../../../; done
```

And finally run a script to interpolate vertical data to pressure levels:

```
cd /g/data/li18/tr2908/aus_tropics_hail/WRF_v4.6.0/simulations/
for i in lat*/WRF/*; do cd $i; echo `pwd`; qsub ~/git/aus_tropics_hail/scripts/interpolate_to_hPa_levels.sh; cd ../../../; done
```

To clean out symlinks to reduce inode usage:

```
cd /g/data/li18/tr2908/aus_tropics_hail/WRF_v4.6.0/simulations/
bash ~/git/aus_tropics_hail/scripts/clean_runs.sh
```

Or

```
for i in lat*/WRF/*; do find $i -maxdepth 1 -type l -exec rm {} +; done
```

To remove the "missing_values" attribute that caused xarray warnings from `basic_*.nc files`, I ran:

```
cd /g/data/li18/tr2908/aus_tropics_hail/WRF_v4.6.0/simulations/
for i in lat*/WRF/*/basic*.nc; do echo $i; ncatted -O -a missing_value,pressure,d,, $i; done
```

## Simulation details

Here are details of the simulation setup, which is simply moved and placed over each hail detection. Three different microphysics schemes are used (NSSL, P3-3M, and MY2). (Wrfout data may be archived in which case these lines are commented out).

In [ ]:
# row = hail_detections.iloc[1,]
# dr = nh.sim_directory(lat=row.latitude, lon=row.longitude, year=row.year, month=row.month,
#                       day=row.day, hour=row.hour, minute=row.minute, sims_dir=sims_dir)
# wm.analyse_wrfinput(glob.glob(f'{dr}/WRF/NSSL/wrfout*')[0])

## Read data

In [ ]:
from importlib import reload
reload(nh)

In [ ]:
spatial_means, spatial_maxes, spatial_mins = nh.read_data(
    hail_detections=hail_detections,
    sims_dir=sims_dir,
    hail_threshold=hail_threshold,
)
results['hail_threshold'] = hail_threshold

In [ ]:
x = spatial_means
pd.Categorical.from_codes(
    x['event_hail_flag'],
    categories=x['event_hail_flag'].attrs['categories']
)

In [ ]:
spatial_means.event_hail_flag.shape

In [ ]:
dict(enumerate(spatial_means.event_hail_flag.attrs['categories']))

In [ ]:
colours=['#EC18DE', '#05A703','#EC18DE', '#05A703']
dict(enumerate(colours))

In [ ]:
reload(nh)
nh.plot_extrema(mins=spatial_mins, maxes=spatial_maxes, file='paper/figures/mins_maxes.pdf')

## Number of hail vs no-hail simulations

In [ ]:
num_simulations = len(spatial_means.event) * len(spatial_means.mp_scheme)
hail_events = np.sum(spatial_means.event_includes_hail).values
nohail_events = np.sum(np.logical_not(spatial_means.event_includes_hail)).values
assert hail_events + nohail_events == num_simulations, 'Error in hail event count'

results['number_mp_schemes'] = len(spatial_means.mp_scheme)
results['hail_cases'] = int(hail_events)
results['nohail_cases'] = int(nohail_events)

print(f'{hail_events}/{num_simulations}\t({np.round(hail_events/num_simulations*100, 1)}%) of the simulations contained surface hail.')
print(f'{nohail_events}/{num_simulations}\t({np.round(nohail_events/num_simulations*100, 1)}%) of the simulations contained no surface hail.')

In [ ]:
nh.plot_hail_simulations(dat=spatial_means, ylim=(-23,-9), xlim=(116, 147), figsize=(10,5), file='paper/figures/sims_map.pdf')

## Hailcast vs microphysics hail sizes

In [ ]:
(results['hailcast_damaging_cases'], results['mp_damaging_cases']) = nh.plot_surface_hailsizes(
    spatial_maxes=spatial_maxes, file='paper/figures/hailsizes.pdf', figsize=(12,3),
)

## Mean skew-T plots by microphysics scheme for hour of hail detection

In [ ]:
nh.skew_T_comp(dat=spatial_means, figsize=(12,3), file='paper/figures/skewT_comp.pdf')

## Mean vertical profiles of hydrometeor mixing ratios for hour of hail detection

In [ ]:
nh.comp_profiles(
    dat=spatial_means,
    factor=1000,
    variables=['qcloud_at_p', 'qrain_at_p', 'qice_at_p', 'qgraup_at_p', 'qhail_at_p'],
    varnames={
        'qcloud_at_p': 'Cloud [g kg$^{-1}$]',
        'qrain_at_p': 'Rain [g kg$^{-1}$]',
        'qice_at_p': 'Ice [g kg$^{-1}$]',
        'qgraup_at_p': 'Graupel [g kg$^{-1}$]',
        'qhail_at_p': 'Hail [g kg$^{-1}$]',
    },
    figsize=(12,11),
    file='paper/figures/hydromets.pdf',
)

## Mean vertical profiles of humidity, wind, and radar reflectivity for hour of hail detection 

In [ ]:
nh.comp_profiles(
    dat=spatial_means,
    variables=['rh_at_p', 'u_at_p', 'v_at_p', 'w_at_p', 'dbz_at_p'],
    varnames={
        'rh_at_p': 'Rel.\nHumidity [%]',
        'u_at_p': 'U wind\n[m s$^{-1}$]',
        'v_at_p': 'V wind\n[m s$^{-1}$]',
        'w_at_p': 'W wind\n[m s$^{-1}$]',
        'dbz_at_p': 'Z\n[dBZ]',
    },
    file='paper/figures/profiles.pdf',
)

## Write results for paper

In [ ]:
with open('paper/results/results.json', 'w') as json_file:
    json.dump(results, json_file, indent=4)